# Open Food Facts — Canadian products (Step 2B)

Stream the 1.27 GB global export, keep ~20 of 211 columns, filter to Canada, and save a clean interim file for enrichment and joins later.

### Setup

Standard imports and project paths. Same pattern as the dunnhumby notebook.

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("../..").resolve()
DATA_RAW = PROJECT_ROOT / "data" / "raw" / "openfoodfacts"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_INTERIM.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Find the OFF file
off_files = list(DATA_RAW.glob("*.csv.gz"))
print(f"Found Open Food Facts files: {[f.name for f in off_files]}")
OFF_FILE = off_files[0]
print(f"File size: {OFF_FILE.stat().st_size / 1e9:.2f} GB")

Found Open Food Facts files: ['en.openfoodfacts.org.products.csv.gz']
File size: 1.27 GB


### Select useful columns (19 of 211)

The file has 211 columns. We only need product identification, category, packaging, country, and basic quality flags. Defining the list up front keeps memory usage low. (`ecoscore_grade` is not present in this export.)

In [10]:
# The 20 columns we actually need from 211
KEEP_COLUMNS = [
    # Product identification
    "code",                    # Barcode / unique ID
    "product_name",            # Product name
    "brands",                  # Brand name
    "brands_tags",             # Standardized brand tag

    # Category — most important for joining to dunnhumby later
    "categories",
    "categories_tags",
    "categories_en",

    # Country filtering
    "countries",
    "countries_tags",
    "countries_en",

    # Physical attributes
    "quantity",                # Package size as a string (e.g. "500 g")
    "packaging",
    "packaging_tags",

    # Quality / nutrition signals
    "labels_tags",             # Organic, fair trade, etc.
    "nutriscore_grade",
    "nova_group",
    # ecoscore_grade not in this export dump

    # Allergens (useful for category enrichment)
    "allergens",
    "main_category",
    "main_category_en",
]

print(f"Will read {len(KEEP_COLUMNS)} columns out of 211 total")  # 19 columns; no ecoscore in this file

Will read 19 columns out of 211 total


### Stream the 1.27 GB file in chunks, keep only Canadian products

We can't load 1.27 GB at once. Read 200K rows at a time, filter each chunk to Canada, and append to a list. At the end, concatenate the filtered chunks.

`countries_tags` contains values like `en:canada,en:united-states`. We filter by checking if "canada" appears in this field.


In [11]:
CHUNK_SIZE = 200_000
canadian_chunks = []
total_rows_read = 0
canadian_rows_found = 0

print(f"Reading {OFF_FILE.name} in chunks of {CHUNK_SIZE:,}...")
print("=" * 60)

reader = pd.read_csv(
    OFF_FILE,
    sep="\t",
    compression="gzip",
    usecols=KEEP_COLUMNS,
    dtype={"code": str},  # barcodes: keep leading zeros, avoid mixed int/str across chunks
    chunksize=CHUNK_SIZE,
    low_memory=False,
    on_bad_lines="skip",
    encoding="utf-8",
)

for i, chunk in enumerate(reader):
    total_rows_read += len(chunk)

    # Filter to Canadian products
    # countries_tags looks like: "en:canada,en:france"
    canada_mask = chunk["countries_tags"].fillna("").str.contains("canada", case=False, na=False)
    canadian = chunk[canada_mask].copy()

    canadian_rows_found += len(canadian)
    canadian_chunks.append(canadian)

    # Progress every 5 chunks
    if (i + 1) % 5 == 0:
        print(f"Chunk {i+1:>3}: read {total_rows_read:>10,} rows | found {canadian_rows_found:>7,} Canadian")

print("=" * 60)
print(f"Done. Total rows scanned: {total_rows_read:,}")
print(f"Canadian products found:   {canadian_rows_found:,}")

# Concatenate all chunks into one dataframe
canadian_products = pd.concat(canadian_chunks, ignore_index=True)
canadian_products["code"] = canadian_products["code"].astype(str)
print(f"\nFinal Canadian products dataframe shape: {canadian_products.shape}")

Reading en.openfoodfacts.org.products.csv.gz in chunks of 200,000...
Chunk   5: read  1,000,000 rows | found  95,953 Canadian
Chunk  10: read  2,000,000 rows | found 111,603 Canadian
Chunk  15: read  3,000,000 rows | found 114,062 Canadian
Chunk  20: read  4,000,000 rows | found 117,923 Canadian
Done. Total rows scanned: 4,527,527
Canadian products found:   120,283

Final Canadian products dataframe shape: (120283, 19)


### Inspect the Canadian products

Check null rates, sample products, and category coverage. Open Food Facts is community-maintained — expect high null rates in most fields.

In [12]:
print("Null rate per column (% missing):")
null_pct = canadian_products.isnull().mean() * 100
print(null_pct.round(1).to_string())

print(f"\nNon-null product names: {canadian_products['product_name'].notna().sum():,}")
print(f"Non-null categories:    {canadian_products['categories_tags'].notna().sum():,}")
print(f"Non-null brands:        {canadian_products['brands'].notna().sum():,}")

print("\nSample of Canadian products with categories:")
sample = canadian_products[
    canadian_products["product_name"].notna() &
    canadian_products["categories_en"].notna()
].sample(10, random_state=42)
print(sample[["product_name", "brands", "categories_en"]].to_string())

Null rate per column (% missing):
code                 0.0
product_name         2.5
quantity            81.2
packaging           96.0
packaging_tags      96.0
brands              40.3
brands_tags         40.1
categories          75.6
categories_tags     75.6
categories_en       75.6
labels_tags         74.7
countries            0.0
countries_tags       0.0
countries_en         0.0
allergens           96.3
nutriscore_grade     0.2
nova_group          84.0
main_category       75.6
main_category_en    75.6

Non-null product names: 117,299
Non-null categories:    29,366
Non-null brands:        71,828

Sample of Canadian products with categories:
                     product_name              brands                                                                                                                                                                                                                                                                 categories_en
14675               Spinac

### Keep only products with at least name + category + barcode

We need rows that have actual information. Products with only a name and nothing else aren't useful for joining or for category enrichment.

In [13]:
before = len(canadian_products)

# Must have product_name, categories_tags, and code (barcode)
useful = canadian_products[
    canadian_products["product_name"].notna() &
    canadian_products["categories_tags"].notna() &
    canadian_products["code"].notna()
].copy()

after = len(useful)
print(f"Canadian products before quality filter: {before:,}")
print(f"Canadian products after quality filter:  {after:,}")
print(f"Retention: {after/before*100:.1f}%")

Canadian products before quality filter: 120,283
Canadian products after quality filter:  28,752
Retention: 23.9%


### Save the Canadian product dataset

Write to Parquet for fast downstream reading.

In [14]:
output_path = DATA_INTERIM / "openfoodfacts_canada.parquet"

# Parquet needs consistent dtypes — OFF mixes int/str/float across chunks
useful_to_save = useful.copy()
useful_to_save["code"] = useful_to_save["code"].astype(str)
useful_to_save["nova_group"] = pd.to_numeric(
    useful_to_save["nova_group"], errors="coerce"
).astype("Int64")

for col in useful_to_save.columns:
    if col in ("code", "nova_group"):
        continue
    useful_to_save[col] = useful_to_save[col].astype("string")

useful_to_save.to_parquet(output_path, index=False)

print(f"Saved: {output_path}")
print(f"File size: {output_path.stat().st_size / 1e6:.1f} MB")
print(f"Rows: {len(useful):,}")
print(f"Columns: {useful.shape[1]}")

Saved: /Users/patel/Documents/E/Data analytics own Project/Freshflow Forecasting system/data/interim/openfoodfacts_canada.parquet
File size: 2.4 MB
Rows: 28,752
Columns: 19


### Count Canadian perishable products by category (for README context)

Quick descriptive analysis of what perishable categories exist in the Canadian OFF data. Output goes into the README as context numbers.


In [15]:
# Look for perishable-related keywords in categories_en
PERISHABLE_KEYWORDS = {
    "DAIRY": ["dairy", "milk", "yogurt", "cheese", "butter", "cream"],
    "MEAT": ["meat", "beef", "pork", "chicken", "poultry", "sausage"],
    "PRODUCE": ["fresh fruit", "fresh vegetable", "produce", "fruits", "vegetables"],
    "BAKERY": ["bread", "bakery", "pastries"],
    "SEAFOOD": ["seafood", "fish", "shellfish"],
    "DELI": ["deli", "prepared", "cooked meat"],
}

useful = pd.read_parquet(DATA_INTERIM / "openfoodfacts_canada.parquet")

print(f"Canadian perishable-related products in Open Food Facts:")
print("=" * 60)
for category, keywords in PERISHABLE_KEYWORDS.items():
    pattern = "|".join(keywords)
    count = useful["categories_en"].str.contains(pattern, case=False, na=False).sum()
    print(f"  {category:10s} {count:>6,}")


Canadian perishable-related products in Open Food Facts:
  DAIRY       3,640
  MEAT        2,062
  PRODUCE     2,412
  BAKERY      1,673
  SEAFOOD       568
  DELI          810
